# FSG-Net on Colab — Retinal Vessel Segmentation

Runs the official FSG-Net (`FSGNet_noGRM` from [`configs/train.yml`](configs/train.yml)) on **DRIVE / STARE / CHASE_DB1 / HRF**.

Paper: [arXiv 2501.18921](https://arxiv.org/abs/2501.18921). Pretrained weights live on the [GitHub release page](https://github.com/ZombaSY/FSG-Net-pytorch/releases/tag/1.1.0).

**Pipeline**
1. Mount Drive, clone repo, install deps
2. Pick a dataset → run that dataset's prep cell **once** (it standardises `img/` + `mask/`)
3. Download the matching pretrained `.pt`, run inference, look at metrics + qualitative output
4. *(Optional)* Train from scratch / fine-tune

**Assumed source layout on Drive** (`MyDrive/dataset/`):
```
DRIVE/      training/{images,1st_manual,mask}  test/{images,1st_manual,2nd_manual,mask}
CHASEDB1/   flat: Image_01L.jpg + Image_01L_1stHO.png
STARE/      images/  snd_label_vk/  1st_labels_ah/  mask/
HRF/        healthy/  glaucoma/  all_2/   (mixed .jpg + .tif)
```
Output layout after prep (under `MyDrive/SegModels/FSGNet/<dataset>/<split>/{img,mask}/`).

## 1. Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO_DIR = '/content/FSG-Net-Baseline'
if not os.path.isdir(REPO_DIR):
    # Option A: clone the public mirror you uploaded to GitHub.
    # Replace with your own fork URL if needed.
    subprocess.run(['git', 'clone', 'https://github.com/ZombaSY/FSG-Net-pytorch.git', REPO_DIR], check=True)
    # Option B: if you have the project mirrored on Drive, copy it instead:
    # !cp -r /content/drive/MyDrive/FSG-Net-Baseline /content/
os.chdir(REPO_DIR)
!ls

In [ ]:
# Colab already has torch, torchvision, numpy, sklearn, opencv, PIL.
# Add the repo-specific extras.
!pip -q install timm==0.9.16 wandb pyyaml scipy

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  |  VRAM: {p.total_memory / 1024**3:.1f} GB')

## 2. Pick dataset + define paths

Set `DATASET` to one of `DRIVE`, `CHASEDB1`, `STARE`, `HRF`. All later cells key off this constant.

In [ ]:
DATASET = 'DRIVE'  # 'DRIVE' | 'CHASEDB1' | 'STARE' | 'HRF'

DRIVE_ROOT = '/content/drive/MyDrive/dataset'        # raw downloads
PREP_ROOT  = '/content/drive/MyDrive/SegModels/FSGNet'  # standardised prep output
CKPT_DIR   = '/content/drive/MyDrive/SegModels/FSGNet/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Paper's per-dataset zero-pad input size (H,W)
INPUT_SIZE = {'DRIVE': 608, 'STARE': 704, 'CHASEDB1': 1024, 'HRF': 1344}[DATASET]

DS_DIR    = f'{PREP_ROOT}/{DATASET.lower()}'
TRAIN_X   = f'{DS_DIR}/train/img'
TRAIN_Y   = f'{DS_DIR}/train/mask'
TEST_X    = f'{DS_DIR}/test/img'
TEST_Y    = f'{DS_DIR}/test/mask'

print(DATASET, '| input_size =', INPUT_SIZE)
print('Prep output dir →', DS_DIR)

## 3. Dataset preparation (run once per dataset)

Each cell reads from `MyDrive/dataset/<NAME>/...` and writes to `MyDrive/SegModels/FSGNet/<name>/{train,test}/{img,mask}/`. After the first run you can skip this section.

Run **only the cell matching `DATASET`**.

In [ ]:
# Shared helpers used by every prep cell.
import shutil, glob
from pathlib import Path
from PIL import Image

def ensure_dirs(*paths):
    for p in paths:
        os.makedirs(p, exist_ok=True)

def save_as_png(src, dst, mode=None):
    """Open arbitrary format (jpg/tif/gif/ppm/png), optionally convert mode, save as PNG."""
    img = Image.open(src)
    if mode is not None:
        img = img.convert(mode)
    img.save(dst, format='PNG')

def binarize_and_save(src, dst, threshold=128):
    img = Image.open(src).convert('L')
    arr = (np.array(img) >= threshold).astype('uint8') * 255
    Image.fromarray(arr).save(dst, format='PNG')

import numpy as np

### 3a. DRIVE

Standard 20 train / 20 test split. Source: `DRIVE/training/{images,1st_manual}` + `DRIVE/test/{images,1st_manual}` (uses `1st_manual` as the GT). If you only have a flat layout, edit the globs below.

In [ ]:
if DATASET == 'DRIVE':
    src = f'{DRIVE_ROOT}/DRIVE'
    splits = {'train': 'training', 'test': 'test'}
    for tgt, sub in splits.items():
        out_x = f'{DS_DIR}/{tgt}/img'
        out_y = f'{DS_DIR}/{tgt}/mask'
        ensure_dirs(out_x, out_y)

        imgs = sorted(glob.glob(f'{src}/{sub}/images/*.tif') + glob.glob(f'{src}/{sub}/images/*.png'))
        gts  = sorted(glob.glob(f'{src}/{sub}/1st_manual/*.gif') + glob.glob(f'{src}/{sub}/1st_manual/*.png') + glob.glob(f'{src}/{sub}/1st_manual/*.tif'))
        assert len(imgs) == len(gts) > 0, f'DRIVE {sub}: imgs={len(imgs)} gts={len(gts)}'

        for img_path, gt_path in zip(imgs, gts):
            stem = Path(img_path).stem.split('_')[0]  # e.g. "21_training" → "21"
            save_as_png(img_path, f'{out_x}/{stem}.png', mode='RGB')
            binarize_and_save(gt_path, f'{out_y}/{stem}.png')

        print(f'DRIVE/{tgt}: {len(imgs)} pairs → {DS_DIR}/{tgt}')
else:
    print('DATASET != DRIVE — skip')

### 3b. CHASE_DB1

Flat folder. Pairs are `Image_XXY.jpg` ↔ `Image_XXY_1stHO.png`. Standard split: first 8 patients (16 images) → train, last 6 (12) → test (per the FR-UNet/FSG-Net convention). Adjust `TRAIN_PATIENTS` if you want a different split.

In [ ]:
if DATASET == 'CHASEDB1':
    src = f'{DRIVE_ROOT}/CHASEDB1'
    TRAIN_PATIENTS = {f'{i:02d}' for i in range(1, 9)}  # 01..08 train
    imgs = sorted([p for p in glob.glob(f'{src}/*.jpg') if 'mask' not in Path(p).stem])
    assert len(imgs) == 28, f'expected 28 CHASE images, found {len(imgs)}'

    for split in ('train', 'test'):
        ensure_dirs(f'{DS_DIR}/{split}/img', f'{DS_DIR}/{split}/mask')

    for img_path in imgs:
        stem = Path(img_path).stem  # Image_01L
        patient = stem.replace('Image_', '')[:2]  # "01"
        split = 'train' if patient in TRAIN_PATIENTS else 'test'
        gt_path = f'{src}/{stem}_1stHO.png'
        assert os.path.exists(gt_path), f'missing GT for {stem}'
        save_as_png(img_path, f'{DS_DIR}/{split}/img/{stem}.png', mode='RGB')
        binarize_and_save(gt_path, f'{DS_DIR}/{split}/mask/{stem}.png')

    for split in ('train', 'test'):
        n = len(glob.glob(f'{DS_DIR}/{split}/img/*.png'))
        print(f'CHASEDB1/{split}: {n} pairs')
else:
    print('DATASET != CHASEDB1 — skip')

### 3c. STARE

Source: `images/im0001.ppm` + `1st_labels_ah/im0001.ah.ppm`. 20 images total; FSG-Net papers typically use a 10/10 split — first 10 → train, last 10 → test.

In [ ]:
if DATASET == 'STARE':
    src = f'{DRIVE_ROOT}/STARE'
    imgs = sorted(glob.glob(f'{src}/images/*.ppm') + glob.glob(f'{src}/images/*.png'))
    assert len(imgs) >= 20, f'expected >=20 STARE images, found {len(imgs)}'

    for split in ('train', 'test'):
        ensure_dirs(f'{DS_DIR}/{split}/img', f'{DS_DIR}/{split}/mask')

    for i, img_path in enumerate(imgs):
        stem = Path(img_path).stem  # im0001
        # match GT by stem prefix
        gt_candidates = glob.glob(f'{src}/1st_labels_ah/{stem}*') + glob.glob(f'{src}/snd_label_vk/{stem}*')
        assert gt_candidates, f'no GT for {stem}'
        gt_path = gt_candidates[0]
        split = 'train' if i < 10 else 'test'
        save_as_png(img_path, f'{DS_DIR}/{split}/img/{stem}.png', mode='RGB')
        binarize_and_save(gt_path, f'{DS_DIR}/{split}/mask/{stem}.png')

    for split in ('train', 'test'):
        n = len(glob.glob(f'{DS_DIR}/{split}/img/*.png'))
        print(f'STARE/{split}: {n} pairs')
else:
    print('DATASET != STARE — skip')

### 3d. HRF

HRF ships 45 images (15 healthy, 15 glaucoma, 15 diabetic). Standard convention: 5 of each class → train (15 total), the other 30 → test. Edit `TRAIN_IDX` if you want a different split.

Expects images under `HRF/all_2/*.jpg` (or category folders) and GT under `HRF/all_2/*.tif` with matching stems. If yours is laid out differently, tweak the globs.

In [ ]:
if DATASET == 'HRF':
    src = f'{DRIVE_ROOT}/HRF'
    # Try the merged folder first, fall back to per-class folders.
    img_pool = sorted(glob.glob(f'{src}/all_2/*.jpg') + glob.glob(f'{src}/all_2/*.JPG'))
    if not img_pool:
        img_pool = sorted(
            glob.glob(f'{src}/healthy/*.jpg') +
            glob.glob(f'{src}/glaucoma/*.jpg') +
            glob.glob(f'{src}/diabetic_retinopathy/*.jpg')
        )
    assert img_pool, f'no HRF images found under {src}'

    TRAIN_IDX = set()  # index 0..14: id 1..5 of each of 3 classes
    for cls_offset in (0, 15, 30):
        TRAIN_IDX.update({cls_offset + i for i in range(5)})

    for split in ('train', 'test'):
        ensure_dirs(f'{DS_DIR}/{split}/img', f'{DS_DIR}/{split}/mask')

    for i, img_path in enumerate(img_pool):
        stem = Path(img_path).stem
        # GT is .tif with same stem; could be in same folder or sibling 'manual1'
        gt_candidates = (
            glob.glob(f'{src}/all_2/{stem}.tif') +
            glob.glob(f'{src}/manual1/{stem}.tif') +
            glob.glob(f'{Path(img_path).parent}/{stem}.tif')
        )
        assert gt_candidates, f'no GT for {stem}'
        split = 'train' if i in TRAIN_IDX else 'test'
        save_as_png(img_path, f'{DS_DIR}/{split}/img/{stem}.png', mode='RGB')
        binarize_and_save(gt_candidates[0], f'{DS_DIR}/{split}/mask/{stem}.png')

    for split in ('train', 'test'):
        n = len(glob.glob(f'{DS_DIR}/{split}/img/*.png'))
        print(f'HRF/{split}: {n} pairs')
else:
    print('DATASET != HRF — skip')

### Sanity-check the prepped data

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample_imgs = sorted(glob.glob(f'{TEST_X}/*.png'))[:3]
fig, axes = plt.subplots(len(sample_imgs), 2, figsize=(8, 3 * len(sample_imgs)))
for i, p in enumerate(sample_imgs):
    img = Image.open(p)
    msk = Image.open(f'{TEST_Y}/{Path(p).stem}.png')
    axes[i, 0].imshow(img); axes[i, 0].set_title(f'img: {Path(p).name}  {img.size}'); axes[i, 0].axis('off')
    axes[i, 1].imshow(msk, cmap='gray'); axes[i, 1].set_title('mask'); axes[i, 1].axis('off')
plt.tight_layout(); plt.show()

## 4. Patch the configs to point at the prepared data

In [ ]:
import yaml, json, textwrap

TRAIN_CFG = f'{REPO_DIR}/configs/train.yml'
INFER_CFG = f'{REPO_DIR}/configs/inference.yml'

# Both yml files are JSON-with-comments; rewrite them as plain JSON so PyYAML loads them.
train_cfg = {
    'debug': False, 'mode': 'train', 'cuda': True, 'pin_memory': True,
    'wandb': False, 'worker': 2, 'log_interval': 9999, 'save_interval': 1,
    'saved_model_directory': f'{CKPT_DIR}/{DATASET.lower()}',
    'train_fold': 1, 'project_name': 'FSG-Net', 'CUDA_VISIBLE_DEVICES': '0',

    'model_name': 'FSGNet_noGRM', 'n_classes': 1, 'in_channels': 3,
    'dataloader': 'Image2Image_zero_pad', 'criterion': 'DiceBCELoss',
    'task': 'segmentation', 'input_space': 'RGB', 'input_channel': 3,
    'input_size': [INPUT_SIZE, INPUT_SIZE],
    'optimizer': 'AdamW', 'lr': 0.001, 'scheduler': 'WarmupCosine',
    'cycles': 100, 'warmup_epoch': 20, 'weight_decay': 0.05,
    'batch_size': 4, 'epoch': 20000, 'ema_decay': 0,
    'class_weight': [1.0, 1.0], 'model_path': '', 'freeze_layer': False,

    'transform_blur': True, 'transform_jitter': True, 'transform_hflip': True,
    'transform_perspective': True, 'transform_cutmix': True,
    'transform_rand_resize': True, 'transform_rand_crop': 288,

    'train_x_path': TRAIN_X, 'train_y_path': TRAIN_Y,
    'val_x_path':   TEST_X,  'val_y_path':   TEST_Y,
}

infer_cfg = {
    'debug': False, 'mode': 'inference', 'cuda': True, 'pin_memory': True,
    'wandb': False, 'worker': 2, 'CUDA_VISIBLE_DEVICES': '0',
    'model_name': 'FSGNet_noGRM', 'n_classes': 1, 'in_channels': 3,
    'inference_mode': 'segmentation', 'task': 'segmentation',
    'dataloader': 'Image2Image_zero_pad', 'criterion': 'DiceBCE',
    'input_space': 'RGB', 'input_channel': 3,
    'input_size': [INPUT_SIZE, INPUT_SIZE],
    'model_path': f'{CKPT_DIR}/FSG-Net-{DATASET}.pt',
    'val_x_path': TEST_X, 'val_y_path': TEST_Y,
}

with open(TRAIN_CFG, 'w') as f: json.dump(train_cfg, f, indent=2)
with open(INFER_CFG, 'w') as f: json.dump(infer_cfg, f, indent=2)
print('Wrote', TRAIN_CFG, 'and', INFER_CFG)

## 5. Inference with pretrained weights

Download the per-dataset checkpoint from the [release page](https://github.com/ZombaSY/FSG-Net-pytorch/releases/tag/1.1.0). The cell below tries the conventional asset URL; if the asset name has changed, just download manually and drop the `.pt` at the path printed by the cell.

In [ ]:
import urllib.request
ckpt_path = infer_cfg['model_path']
release = 'https://github.com/ZombaSY/FSG-Net-pytorch/releases/download/1.1.0'
asset_name = f'FSG-Net-{DATASET}.pt'  # adjust if release uses a different filename
url = f'{release}/{asset_name}'

if not os.path.exists(ckpt_path):
    print('Downloading', url, '→', ckpt_path)
    try:
        urllib.request.urlretrieve(url, ckpt_path)
    except Exception as e:
        print('Auto-download failed:', e)
        print('→ Open the release page manually, save the file to:', ckpt_path)
else:
    print('Checkpoint already present:', ckpt_path)
print('Size (MB):', os.path.getsize(ckpt_path) / 1024**2 if os.path.exists(ckpt_path) else 'N/A')

In [ ]:
# Run inference. Should print mIoU / F1 / Acc / AUC / Sensitivity / MCC.
!cd {REPO_DIR} && python3 main.py --config_path configs/inference.yml

### Visualise predictions

`inference.py` writes `<imgname>_argmax.png` next to the checkpoint, under a folder named after the model file. Show a few alongside ground truth.

In [ ]:
pred_dir = f'{CKPT_DIR}/FSG-Net-{DATASET}/'  # follows inference.py naming
preds = sorted(glob.glob(f'{pred_dir}*_argmax.png'))[:6]
if not preds:
    print('No predictions found at', pred_dir)
else:
    fig, axes = plt.subplots(len(preds), 3, figsize=(10, 3 * len(preds)))
    if len(preds) == 1: axes = axes[None, :]
    for i, p in enumerate(preds):
        # Pred filename: <stem>_<idx>_argmax.png — strip the appended idx + suffix.
        stem = Path(p).stem.replace('_argmax', '')
        stem = '_'.join(stem.split('_')[:-1])  # drop trailing '_<idx>'
        img_p = f'{TEST_X}/{stem}.png'
        gt_p  = f'{TEST_Y}/{stem}.png'
        if os.path.exists(img_p):
            axes[i, 0].imshow(Image.open(img_p)); axes[i, 0].set_title(stem); axes[i, 0].axis('off')
        if os.path.exists(gt_p):
            axes[i, 1].imshow(Image.open(gt_p), cmap='gray'); axes[i, 1].set_title('GT'); axes[i, 1].axis('off')
        axes[i, 2].imshow(Image.open(p), cmap='gray'); axes[i, 2].set_title('pred'); axes[i, 2].axis('off')
    plt.tight_layout(); plt.show()

## 6. (Optional) Train from scratch / fine-tune

Default config matches the paper (batch 4, AdamW, WarmupCosine, 288² random crop). To fine-tune from a pretrained checkpoint, set `train_cfg['model_path']` to the `.pt` path and re-run the patch cell.

Notes for L4/A100 Colab Pro:
- DRIVE/STARE/CHASE: defaults fit comfortably.
- HRF: validation forward pass at 1344² is heavy. If you OOM, drop `input_size` to e.g. `[1024, 1024]` (still > image size after pad) or run inference on CPU.
- Training crop is 288² regardless of `input_size`, so the crop's VRAM cost is constant — `input_size` only affects validation.

In [ ]:
# Uncomment to train.
# !cd {REPO_DIR} && python3 main.py --config_path configs/train.yml